# Dominick weekly panel for ICDN

Step-by-step mapping of the observation unit

`(store, UPC, week) → (price per liter, liters sold, observed promo)`

We do **not** call `builder.run()`. Each cell invokes one method so we can inspect the intermediate objects.

**File used:** `data/Dominick/dominick_features.csv` (already weekly).  
**Not reconstructed:** demand, prices, or promotions.

**ICDN 1.0.0:** `units > 0` and `price > 0`. Placeholder rows with zero pack counts also have zero price and zero liters; they are dropped, never recoded as ones.

`on_promo` is the **observed** Dominick flag (B/S/C), not a markdown proxy.

Quantity is **liters sold** and price is **price per liter**, so 6-packs and 24-packs are comparable. ICDN logs both internally — do not pass `log_liters_sold` / `log_price_per_liter`.

ICDN elasticity bounds for this panel are set in `ICDN_EXTRAS["dominick"]`: own `[-5, 0]`, cross `[-1, 1]`.


In [1]:
from pathlib import Path
import sys

import pandas as pd

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from dominick import CATEGORY_LABELS, SOURCE_COLUMNS, DominickConfig, DominickPanelBuilder

DATA_DIR = PROJECT_ROOT / "data" / "Dominick"
OUT_DIR = DATA_DIR / "panel"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_DIR    :", DATA_DIR)
print("Exists      :", DATA_DIR.exists())
print("Files       :", sorted(p.name for p in DATA_DIR.glob("*")) if DATA_DIR.exists() else "—")
print("CATEGORY_LABELS:", CATEGORY_LABELS)


PROJECT_ROOT: /home/thebigmonster/Github/nn-elasticity-additional-work
DATA_DIR    : /home/thebigmonster/Github/nn-elasticity-additional-work/data/Dominick
Exists      : True
Files       : ['dominick_features.csv', 'panel']
CATEGORY_LABELS: {26: 'imported_beer', 27: 'beer', 28: 'nonalcoholic_beer'}


## 0. Config and builder

Selection rules are stored now but applied only on the **first half** of observed weeks.  
`target_category_id` stays `None` until we inspect `category_stats`.


In [2]:
config = DominickConfig(
    data_dir=DATA_DIR,
    out_dir=OUT_DIR,
    selection_frac=0.50,
    min_store_week_coverage=0.85,
    n_core_stores=20,
    min_stores=10,
    min_week_coverage=0.70,
    min_unique_prices=8,
    min_price_cv=0.03,
    n_candidate_skus=40,
    n_skus=20,
    target_category_id=None,
)

builder = DominickPanelBuilder(config)


## 1. Source columns

The CSV has many precomputed fields. Only the ICDN inputs (plus description for audit) are loaded. Logs, promo subtype dummies, pack counts, shelf prices, and dollar margins stay on disk.


In [3]:
all_cols = pd.read_csv(DATA_DIR / "dominick_features.csv", nrows=0).columns.tolist()
used = set(SOURCE_COLUMNS)
inventory_source = pd.DataFrame({
    "column": all_cols,
    "loaded": [c in used for c in all_cols],
})
inventory_source


,column,loaded
0,category_code,True
1,upc_code,True
2,product_description,True
3,pack_size_text,True
4,units_per_case,False
5,product_item_code,False
6,store_code,True
7,week_id,True
8,units_sold,True
9,units_per_deal,False


## 2. Load the feature table


In [4]:
raw = builder.load_features()
print(raw.shape)
raw.head()


source dominick_features.csv (3967720, 13)
columns used: ['category_code', 'upc_code', 'product_description', 'pack_size_text', 'store_code', 'week_id', 'units_sold', 'unit_price', 'liters_sold', 'price_per_liter', 'on_promo', 'brand_family_norm', 'style_segment_norm']
(3967720, 13)


,category_code,upc_code,product_description,pack_size_text,store_code,week_id,units_sold,unit_price,brand_family_norm,style_segment_norm,liters_sold,price_per_liter,on_promo
0,27,294,BEER LIMIT,12/12O,2,298,11,2.6200,BEER_LIMIT,UNKNOWN_STYLE,46.8445,0.6152,False
1,27,294,BEER LIMIT,12/12O,2,299,9,2.5100,BEER_LIMIT,UNKNOWN_STYLE,38.3273,0.5894,False
2,27,294,BEER LIMIT,12/12O,2,300,6,2.4900,BEER_LIMIT,UNKNOWN_STYLE,25.5515,0.5847,False
3,27,294,BEER LIMIT,12/12O,2,301,5,2.9900,BEER_LIMIT,UNKNOWN_STYLE,21.2929,0.7021,False
4,27,294,BEER LIMIT,12/12O,2,302,1,2.4900,BEER_LIMIT,UNKNOWN_STYLE,4.2586,0.5847,False


## 3. Column map

ICDN needs `store_code`, `product_code`, `week_id`, `price`, `units`, `on_promo`. Brand, style, and pack size bias competitor selection.


In [5]:
inventory = builder.column_inventory(raw)
inventory


             column   role                                                         icdn
      category_code mapped                               category (via CATEGORY_LABELS)
           upc_code mapped                                                 product_code
product_description unused                                     audit / diagnostics only
     pack_size_text mapped                                                         size
         store_code mapped                                                   store_code
            week_id mapped week_id (compacted to 1..T; original kept as source_week_id)
         units_sold unused          pack-count demand; not comparable across pack sizes
         unit_price unused              shelf price per pack; ICDN uses price per liter
  brand_family_norm mapped                                                        brand
 style_segment_norm mapped                                                        style
        liters_sold mapped      

,column,role,icdn
0,category_code,mapped,category (via CATEGORY_LABELS)
1,upc_code,mapped,product_code
2,product_description,unused,audit / diagnostics only
3,pack_size_text,mapped,size
4,store_code,mapped,store_code
5,week_id,mapped,week_id (compacted to 1..T; original kept as s...
6,units_sold,unused,pack-count demand; not comparable across pack ...
7,unit_price,unused,shelf price per pack; ICDN uses price per liter
8,brand_family_norm,mapped,brand
9,style_segment_norm,mapped,style


## 4. Map to the ICDN schema

- `product_code` = UPC (`upc_code`), not `product_item_code` (one item code can cover several UPCs).
- `units` = `liters_sold`, `price` = `price_per_liter`.
- Placeholder rows are dropped.
- Source week codes have calendar holes; they are remapped to consecutive `week_id ∈ {1,…,T}`. The original code is `source_week_id`.


In [6]:
weekly = builder.to_icdn_schema(raw)
print(weekly.dtypes)
print(weekly.groupby("category", observed=True).agg(
    n_upc=("product_code", "nunique"),
    n_rows=("units", "size"),
    promo_rate=("on_promo", "mean"),
))
weekly.head()


placeholder rows (units<=0 or liters<=0 or price<=0): 2001572
source weeks 91 → 399 n= 302 n_gaps 3 max_gap 5
observed rows 1966148 stores 89 UPCs 787 weeks 302
store_code             string[python]
product_code           string[python]
week_id                         int32
source_week_id                  int32
price                         float32
units                         float32
on_promo                         int8
category_id                     int32
category               string[python]
brand                  string[python]
style                  string[python]
size                   string[python]
product_description    string[python]
dtype: object
                   n_upc   n_rows  promo_rate
category                                     
beer                 586  1460607      0.2712
imported_beer        160   363067      0.3123
nonalcoholic_beer     41   142474      0.2477


,store_code,product_code,week_id,source_week_id,price,units,on_promo,category_id,category,brand,style,size,product_description
0,100,1820000008,1,91,1.6801,13.2489,0,27,beer,BUDWEISER,UNKNOWN_STYLE,32 OZ,BUDWEISER BEER N.R.B
1,100,1820000016,1,91,1.6390,104.3354,0,27,beer,BUDWEISER,UNKNOWN_STYLE,6/12 O,BUDWEISER BEER
2,100,1820000018,1,91,1.6167,31.2296,0,27,beer,BUDWEISER,UNKNOWN_STYLE,6/16 O,BUDWEISER BEER
3,100,1820000106,1,91,1.7330,4.2586,0,27,beer,BUDWEISER,LIGHT,6/12 O,BUDWEISER LIGHT BEER
4,100,1820000202,1,91,1.7330,8.5172,0,27,beer,BUDWEISER,UNKNOWN_STYLE,6/12 O,BUDWEISER DRY BEER


## 5. Save the weekly master

Scientific file: observed positive store–UPC–weeks only. No invented zeros.


In [7]:
weekly = builder.save_weekly_master(weekly)
print(weekly.shape)
print(
    "stores", weekly.store_code.nunique(),
    "UPCs", weekly.product_code.nunique(),
    "weeks", weekly.week_id.nunique(),
)


Wrote /home/thebigmonster/Github/nn-elasticity-additional-work/data/Dominick/panel/dominick_weekly_master.parquet
(1966148, 13)
stores 89 UPCs 787 weeks 302


## 6. Selection window (no lookahead)

SKU and store choice uses only `week_id <= cutoff`. The same frozen list is then used on the full horizon.


In [8]:
selection = builder.selection_sample()
print(selection.shape)


Selection cutoff: 151 of 1 → 302
selection weeks: 151
(1097505, 13)


## 7. Core stores

Keep stores that cover at least `min_store_week_coverage` of the selection window, then the top `n_core_stores` by observation count.


In [9]:
core_stores = builder.select_core_stores(selection)
builder.store_stats.sort_values("week_coverage", ascending=False).head(15)


core stores: ['128', '101', '126', '122', '131', '100', '103', '112', '121', '129', '132', '98', '134', '105', '102', '109', '115', '12', '71', '32']


,store_code,n_obs,n_items,n_weeks,total_units,week_coverage
0,100,22455,343,151,"872,611.2500",1.0000
1,101,23459,338,151,"918,743.3750",1.0000
2,102,20633,315,151,"904,931.6250",1.0000
3,103,22272,323,151,"1,044,910.0000",1.0000
4,105,20701,326,151,"830,288.7500",1.0000
5,106,14210,269,151,"335,417.1875",1.0000
7,109,20466,299,151,"524,944.3750",1.0000
10,112,22239,303,151,"721,563.3750",1.0000
9,111,9057,168,151,"175,097.4219",1.0000
58,78,11872,215,151,"357,152.1875",1.0000


## 8. Product diagnostics (core stores, selection window)


In [10]:
selection_core = selection[selection["store_code"].isin(core_stores)]
product_stats = builder.compute_product_stats(selection_core)
product_stats.sort_values("coverage_rate", ascending=False).head(15)


Wrote /home/thebigmonster/Github/nn-elasticity-additional-work/data/Dominick/panel/dominick_product_diagnostics.csv


,product_code,category_id,category,brand,style,size,product_description,n_obs,n_stores,n_weeks,total_units,unique_prices,mean_price,std_price,promo_rate,price_cv,coverage_rate
94,3410000354,27,beer,MILLER,LIGHT,6/12 O,MILLER LITE BEER,3014,20,151,"114,251.5391",57,1.7574,0.0791,0.0036,0.0450,0.9980
147,3410057306,27,beer,MILLER,LIGHT,24/12O,MILLER LITE BEER,3013,20,151,"1,711,347.7500",109,1.3092,0.0825,0.4935,0.0631,0.9977
123,3410017306,27,beer,MILLER,DRAFT,24/12O,MILLER GENUINE DRAFT,3012,20,151,"1,028,176.5000",103,1.3092,0.0825,0.4844,0.0630,0.9974
124,3410017505,27,beer,MILLER,DRAFT,6/12 O,MILLER GEN DRFT LNNR,3012,20,151,"242,564.9219",75,1.7108,0.1890,0.3127,0.1105,0.9974
148,3410057505,27,beer,MILLER,LIGHT,6/12 O,MILLER LITE LONGNECK,3010,20,151,"189,982.0156",77,1.7135,0.1875,0.3120,0.1094,0.9967
48,1820011168,27,beer,BUDWEISER,UNKNOWN_STYLE,24/12O,BUDWEISER BEER,3006,20,151,"646,990.2500",126,1.3169,0.0820,0.4584,0.0623,0.9954
152,3410057602,27,beer,MILLER,LIGHT,12/12O,MILLER LITE BEER,3006,20,151,"311,537.0312",110,1.6228,0.1556,0.2272,0.0959,0.9954
116,3410015306,27,beer,MILLER,DRAFT,24/12O,MILLER GENUINE DRFT,3000,20,151,"524,189.6250",97,1.3091,0.0827,0.4907,0.0631,0.9934
1,1820000016,27,beer,BUDWEISER,UNKNOWN_STYLE,6/12 O,BUDWEISER BEER,3000,20,151,"85,595.4922",38,1.7556,0.0763,0.0000,0.0434,0.9934
113,3410010505,28,nonalcoholic_beer,MILLER,NON_ALCOHOLIC,6/12OZ,MILLER SHARP'S N/A L,2991,20,151,"99,240.0078",55,1.7145,0.1779,0.3220,0.1037,0.9904


## 9. Eligible screen

Density, store coverage, unique prices, and price CV. Applied on the selection window only.


In [11]:
eligible = builder.screen_eligible(product_stats)
print(eligible.groupby("category", observed=True).size())
eligible[
    ["product_code", "product_description", "category", "brand", "style", "size",
     "coverage_rate", "n_stores", "n_weeks", "unique_prices", "price_cv", "promo_rate"]
].head(20)


eligible SKUs: 161
category
beer                 134
imported_beer         21
nonalcoholic_beer      6
dtype: int64


,product_code,product_description,category,brand,style,size,coverage_rate,n_stores,n_weeks,unique_prices,price_cv,promo_rate
94,3410000354,MILLER LITE BEER,beer,MILLER,LIGHT,6/12 O,0.9980,20,151,57,0.0450,0.0036
147,3410057306,MILLER LITE BEER,beer,MILLER,LIGHT,24/12O,0.9977,20,151,109,0.0631,0.4935
123,3410017306,MILLER GENUINE DRAFT,beer,MILLER,DRAFT,24/12O,0.9974,20,151,103,0.0630,0.4844
124,3410017505,MILLER GEN DRFT LNNR,beer,MILLER,DRAFT,6/12 O,0.9974,20,151,75,0.1105,0.3127
148,3410057505,MILLER LITE LONGNECK,beer,MILLER,LIGHT,6/12 O,0.9967,20,151,77,0.1094,0.3120
48,1820011168,BUDWEISER BEER,beer,BUDWEISER,UNKNOWN_STYLE,24/12O,0.9954,20,151,126,0.0623,0.4584
152,3410057602,MILLER LITE BEER,beer,MILLER,LIGHT,12/12O,0.9954,20,151,110,0.0959,0.2272
116,3410015306,MILLER GENUINE DRFT,beer,MILLER,DRAFT,24/12O,0.9934,20,151,97,0.0631,0.4907
1,1820000016,BUDWEISER BEER,beer,BUDWEISER,UNKNOWN_STYLE,6/12 O,0.9934,20,151,38,0.0434,0.0000
113,3410010505,MILLER SHARP'S N/A L,nonalcoholic_beer,MILLER,NON_ALCOHOLIC,6/12OZ,0.9904,20,151,55,0.1037,0.3220


## 10. Rank categories — STOP

Inspect before freezing.

| `category_id` | label |
| --- | --- |
| 26 | imported beer |
| 27 | beer |
| 28 | non-alcoholic beer |

Do not pick 28 if the question is alcoholic beer demand. Ranking by median coverage can put imported beer (26) first; the research question here is **beer** (27), the largest eligible set. Do not revise the choice after seeing ICDN vs MLP.


In [12]:
category_stats = builder.rank_categories(eligible)
category_stats


 category_id      category  n_products  median_coverage  median_price_cv  median_n_stores     total_units
          27          beer         134           0.7478           0.0776          20.0000 11,868,859.0000
          26 imported_beer          21           0.5964           0.1097          20.0000    471,085.0312


,category_id,category,n_products,median_coverage,median_price_cv,median_n_stores,total_units
1,27,beer,134,0.7478,0.0776,20.0000,"11,868,859.0000"
0,26,imported_beer,21,0.5964,0.1097,20.0000,"471,085.0312"


## 11. Freeze 20 SKUs by Jaccard co-occurrence

Top 40 eligible UPCs in category 27; greedy Jaccard + coverage picks 20 that share store-weeks (so the Jacobian is identified).


In [13]:
# Set AFTER inspecting category_stats. 28 is not a default. 26 is imported beer.
TARGET_CATEGORY_ID = 27
print(
    "Using category_id =", TARGET_CATEGORY_ID,
    CATEGORY_LABELS.get(TARGET_CATEGORY_ID),
)
if TARGET_CATEGORY_ID not in set(category_stats["category_id"].astype(int)):
    raise ValueError(
        f"category_id={TARGET_CATEGORY_ID} is not in category_stats. "
        "Inspect the ranking before freezing."
    )

selection = builder.freeze_universe(TARGET_CATEGORY_ID)
selection.to_dict()


Using category_id = 27 beer
SELECTED_PRODUCTS: ['3410000354', '3410057306', '3410017306', '3410017505', '3410057505', '1820011168', '3410057602', '3410015306', '1820000016', '3410017528', '3410017602', '3410000554', '7204001113', '3410015505', '3410057528', '1820000834', '1820011047', '1820000987', '7336011751', '1820053168']
count   3,014.0000
mean        0.9899
std         0.0264
min         0.6000
25%         1.0000
50%         1.0000
75%         1.0000
max         1.0000
dtype: float64
store-weeks >= 16/20: 0.9996682149966821
store-weeks = 20/20: 0.8424021234240212
Wrote frozen store/SKU lists under /home/thebigmonster/Github/nn-elasticity-additional-work/data/Dominick/panel


{'cutoff_week_id': 151,
 'category_id': 27,
 'category': 'beer',
 'core_stores': ['128',
  '101',
  '126',
  '122',
  '131',
  '100',
  '103',
  '112',
  '121',
  '129',
  '132',
  '98',
  '134',
  '105',
  '102',
  '109',
  '115',
  '12',
  '71',
  '32'],
 'candidate_product_codes': ['3410000354',
  '3410057306',
  '3410017306',
  '3410017505',
  '3410057505',
  '1820011168',
  '3410057602',
  '3410015306',
  '1820000016',
  '3410017528',
  '3410017602',
  '3410000554',
  '7204001113',
  '3410015505',
  '1820000834',
  '3410057528',
  '1820011047',
  '1820000987',
  '7336011751',
  '1820053168',
  '3410017525',
  '3410021505',
  '3410057525',
  '3410015602',
  '7336011341',
  '1820000833',
  '1820061168',
  '7204004443',
  '7336011661',
  '7199031600',
  '3410000904',
  '7199000048',
  '1820000106',
  '3410001505',
  '3410015525',
  '8417330130',
  '3410015528',
  '1820053047',
  '7336011754',
  '1820086167'],
 'product_codes': ['3410000354',
  '3410057306',
  '3410017306',
  '3410017

## 12. ICDN panel

Full horizon, frozen stores and SKUs. Columns: `store_code`, `product_code`, `week_id`, `price`, `units`, `on_promo`, `category`, `brand`, `style`, `size`.


In [14]:
icdn_panel = builder.build_icdn_panel()
print(icdn_panel.shape)
print(icdn_panel.dtypes)
icdn_panel.head()


Wrote /home/thebigmonster/Github/nn-elasticity-additional-work/data/Dominick/panel/dominick_icdn_panel.parquet
              n_obs  n_weeks  n_stores  mean_units
product_code                                      
1820000016     4342      220        20     26.3886
1820000834     4330      220        20     27.1635
1820000987     4279      220        20     21.9135
1820011047     4317      220        20     42.8394
1820011168     4385      220        20    207.0286
1820053168     4302      220        20     83.5463
3410000354     4383      220        20     33.8239
3410000554     4335      220        20     18.4470
3410015306     4361      220        20    165.6738
3410015505     4324      220        20     34.9216
3410017306     4392      220        20    328.1867
3410017505     4391      220        20     74.6785
3410017528     4363      220        20    131.3125
3410017602     4345      220        20     58.2040
3410057306     4393      220        20    559.1944
3410057505     4390   

,store_code,product_code,week_id,price,units,on_promo,category,brand,style,size
0,100,1820000016,1,1.6390,104.3354,0,beer,BUDWEISER,UNKNOWN_STYLE,6/12 O
1,100,1820000834,1,1.7330,29.8101,0,beer,BUDWEISER,UNKNOWN_STYLE,6/12 O
2,100,1820000987,1,1.5451,127.7576,1,beer,MICHELOB,REGULAR,6/12 O
3,100,1820011047,1,1.5944,76.6546,0,beer,BUDWEISER,UNKNOWN_STYLE,12/12O
4,100,1820011168,1,1.1729,"1,294.6108",0,beer,BUDWEISER,UNKNOWN_STYLE,24/12O


## 13. Final checks

No duplicate keys, strictly positive price and units, binary promo. `week_id` is increasing (calendar holes among the frozen SKUs stay holes; we do not re-compact).


In [15]:
assert (icdn_panel["price"] > 0).all()
assert (icdn_panel["units"] > 0).all()
assert set(icdn_panel["on_promo"].unique()).issubset({0, 1})
assert not icdn_panel.duplicated(["store_code", "product_code", "week_id"]).any()
weeks = sorted(icdn_panel["week_id"].unique())
assert weeks[0] == min(weeks)
print("OK")
print(
    "n_stores", icdn_panel.store_code.nunique(),
    "n_skus", icdn_panel.product_code.nunique(),
    "n_weeks", icdn_panel.week_id.nunique(),
    "promo_rate", float(icdn_panel.on_promo.mean()),
)
print(OUT_DIR)
print(sorted(p.name for p in OUT_DIR.glob("*") if p.suffix in {".parquet", ".csv", ".json"}))


OK
n_stores 20 n_skus 20 n_weeks 220 promo_rate 0.276897731592954
/home/thebigmonster/Github/nn-elasticity-additional-work/data/Dominick/panel
['dominick_category_diagnostics.csv', 'dominick_icdn_panel.parquet', 'dominick_product_diagnostics.csv', 'dominick_selected_products.csv', 'dominick_selected_skus.json', 'dominick_selected_stores.csv', 'dominick_store_diagnostics.csv', 'dominick_weekly_master.parquet']
